<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    X, y = fetch_openml('Fashion-MNIST', version=1, as_frame=False, return_X_y=True)
    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

Não é preciso normalizar os dados ao utilizar Random Forest e AdaBoost, pois ambos são baseados em árvores de decisão. Esses modelos não utilizam medidas de distância ou operações sensíveis à escala. Dessa forma, a magnitude dos valores não influencia o processo de aprendizado, e a normalização não impacta significativamente o desempenho do modelo.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)
    
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(n_estimators=50, random_state=seed)
    model.fit(X_train, y_train)
    
    return model


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [3]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    # Fazer predições
    y_pred = model.predict(X_test)
    
    # Calcular acurácia
    acc = accuracy_score(y_test, y_pred)
    
    return acc

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    # 1. Carregar dados
    X_train, X_test, y_train, y_test = load_data(seed)
    
    # 2. Escolher modelo
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    # 3. Avaliar modelo
    acc = evaluate(model, X_test, y_test)
    
    return acc

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Carregar dados
X_train, X_test, y_train, y_test = load_data(seed=42)

print("Profundidade | Treino | Teste")
print("-" * 35)

for d in range(1, 21):
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    
    # treino
    train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, train_pred)
    
    # teste
    test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, test_pred)
    
    print(f"{d:11d} | {train_acc:.4f} | {test_acc:.4f}")

Profundidade | Treino | Teste
-----------------------------------
          1 | 0.1993 | 0.1990
          2 | 0.3565 | 0.3560
          3 | 0.5019 | 0.5016
          4 | 0.6538 | 0.6479
          5 | 0.7105 | 0.7023
          6 | 0.7372 | 0.7239
          7 | 0.7751 | 0.7605
          8 | 0.8033 | 0.7814
          9 | 0.8267 | 0.7966
         10 | 0.8481 | 0.8050
         11 | 0.8676 | 0.8138
         12 | 0.8874 | 0.8143
         13 | 0.9069 | 0.8153
         14 | 0.9256 | 0.8164
         15 | 0.9406 | 0.8140
         16 | 0.9549 | 0.8109
         17 | 0.9659 | 0.8089
         18 | 0.9745 | 0.8064
         19 | 0.9802 | 0.8066
         20 | 0.9845 | 0.8026


Na profundidade 15.

Pois quando max_depth=None, a árvore cresce até que cada folha contenha amostras puras, o que pode levar a uma acurácia de 100% no conjunto de treino, pois o modelo memoriza os dados em vez de aprender padrões generalizáveis.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Carregar dados
X_train, X_test, y_train, y_test = load_data(seed=42)

#Random Forest
rf_model = train_random_forest(X_train, y_train, seed=42)

rf_acc = evaluate(rf_model, X_test, y_test)
rf_pred = rf_model.predict(X_test)

rf_precision = precision_score(y_test, rf_pred, average='macro')
rf_recall = recall_score(y_test, rf_pred, average='macro')
rf_f1 = f1_score(y_test, rf_pred, average='macro')


#AdaBoost
ab_model = train_adaboost(X_train, y_train, seed=42)

ab_acc = evaluate(ab_model, X_test, y_test)
ab_pred = ab_model.predict(X_test)

ab_precision = precision_score(y_test, ab_pred, average='macro')
ab_recall = recall_score(y_test, ab_pred, average='macro')
ab_f1 = f1_score(y_test, ab_pred, average='macro')

print("\nRandom Forest")
print(f"Acurácia:  {rf_acc:.4f}")
print(f"Precisão:  {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-Score:  {rf_f1:.4f}")

print("\nAdaBoost")
print(f"Acurácia:  {ab_acc:.4f}")
print(f"Precisão:  {ab_precision:.4f}")
print(f"Recall:    {ab_recall:.4f}")
print(f"F1-Score:  {ab_f1:.4f}")


Random Forest
Acurácia:  0.8819
Precisão:  0.8807
Recall:    0.8819
F1-Score:  0.8800

AdaBoost
Acurácia:  0.4909
Precisão:  0.4717
Recall:    0.4909
F1-Score:  0.4448


O Random Forest teve um desempenho melhor no modelo inicial

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [7]:
# Seed 42
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model_42 = train_random_forest(X_train, y_train, seed=42)
rf_acc_42 = evaluate(rf_model_42, X_test, y_test)

ab_model_42 = train_adaboost(X_train, y_train, seed=42)
ab_acc_42 = evaluate(ab_model_42, X_test, y_test)


# Seed 7
X_train, X_test, y_train, y_test = load_data(seed=7)

rf_model_7 = train_random_forest(X_train, y_train, seed=7)
rf_acc_7 = evaluate(rf_model_7, X_test, y_test)

ab_model_7 = train_adaboost(X_train, y_train, seed=7)
ab_acc_7 = evaluate(ab_model_7, X_test, y_test)


print("Random Forest (seed=42):", rf_acc_42)
print("Random Forest (seed=7):", rf_acc_7)

print("AdaBoost (seed=42):", ab_acc_42)
print("AdaBoost (seed=7):", ab_acc_7)

Random Forest (seed=42): 0.8818571428571429
Random Forest (seed=7): 0.8800714285714286
AdaBoost (seed=42): 0.49092857142857144
AdaBoost (seed=7): 0.45535714285714285


Os dois mudaram, porém no random forest mudou muito pouco e no AdaBoost apresentou uma variação mais significativa.

O experimento é reprodutível, pois ao utilizar a mesma seed os resultados permanecem consistentes entre execuções. O uso do parâmetro random_state garante que os processos aleatórios sejam controlados, permitindo reproduzir exatamente os mesmos resultados.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [10]:
from sklearn.metrics import accuracy_score

#Random Forst 
# Treino
rf_train_pred = rf_model.predict(X_train)
rf_train_acc = accuracy_score(y_train, rf_train_pred)

# Teste
rf_test_acc = evaluate(rf_model, X_test, y_test)

print("Random Forest")
print(f"Acurácia treino: {rf_train_acc:.4f}")
print(f"Acurácia teste:  {rf_test_acc:.4f}")

#AdaBoost
# Treino
ab_train_pred = ab_model.predict(X_train)
ab_train_acc = accuracy_score(y_train, ab_train_pred)

# Teste
ab_test_acc = evaluate(ab_model, X_test, y_test)

print("\nAdaBoost")
print(f"Acurácia treino: {ab_train_acc:.4f}")
print(f"Acurácia teste:  {ab_test_acc:.4f}")

Random Forest
Acurácia treino: 1.0000
Acurácia teste:  0.8819

AdaBoost
Acurácia treino: 0.4951
Acurácia teste:  0.4909


O random forest está sofrendo mais com overfitting, já que no treino ela está com uma acurácia bem maior que no teste.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [11]:
#Random Forest
rf_results = []

for n in [10, 50, 100]:
    model = train_random_forest(X_train, y_train, seed=42)
    model.n_estimators = n  
    model.fit(X_train, y_train)
    
    acc = evaluate(model, X_test, y_test)
    rf_results.append((n, acc))

print("Random Forest:", rf_results)

#AdaBoost
ab_results = []

for n in [10, 50, 100]:
    from sklearn.ensemble import AdaBoostClassifier
    
    model = AdaBoostClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    
    acc = evaluate(model, X_test, y_test)
    ab_results.append((n, acc))

print("AdaBoost:", ab_results)

Random Forest: [(10, 0.8613571428571428), (50, 0.878), (100, 0.8818571428571429)]
AdaBoost: [(10, 0.31314285714285717), (50, 0.49092857142857144), (100, 0.5728571428571428)]


Ao variar o número de estimadores, observou-se que o desempenho do Random Forest apresentou apenas pequenas melhorias. Já o AdaBoost apresentou variações mais significativas no desempenho, com melhorias mais evidentes ao aumentar o número de estimadores.

O AdaBoost é mais sensível a mudança.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1 - Não, a acurácia sozinha não é suficiente, especialmente em problemas de classificação multiclasse como o Fashion MNIST. Embora ela indique a proporção de acertos, não revela como o modelo se comporta em cada classe individualmente. Por isso, métricas como precisão, recall e F1-score são importantes, pois fornecem uma visão mais completa do desempenho, especialmente em cenários onde pode haver confusão entre classes semelhantes.

2 - O controle da aleatoriedade através do uso de uma seed (random_state) garante que os resultados sejam reprodutíveis, ou seja, ao executar o experimento novamente com a mesma seed, os resultados serão os mesmos. Além disso, ao testar com as seeds 7 e 42 e observar pequenas variações nos resultados, é possível verificar que o comportamento do modelo é consistente e não depende de uma única divisão específica dos dados.

3 - Um possível problema é a avaliação baseada em apenas uma divisão treino/teste, o que pode introduzir viés dependendo da forma como os dados foram separados. Uma abordagem mais robusta seria utilizar validação cruzada. Outro problema é a ausência de ajuste de hiperparâmetros, o que pode levar a modelos subótimos, como observado no desempenho inferior do AdaBoost, que poderia ser melhorado com o ajuste do hiperparâmetro adequado.

4 - O pipeline pode ser considerado confiável, pois segue boas práticas como separação adequada entre treino e teste, uso de seeds para reprodutibilidade e modularização das etapas (carregamento, treinamento e avaliação). No entanto, sua confiabilidade poderia ser aumentada com o uso de validação cruzada e uma análise mais aprofundada dos hiperparâmetros, reduzindo a dependência de uma única configuração experimental.